# PicoCal - Minimum-bias: two-target study (notebook 13)



In [1]:
import sys, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import HistGradientBoostingRegressor

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
sys.path.insert(0, str(repo / "scripts"))
from run_experiments import build, select_knn, split, resolution, EPS

SEEDS = 3
cfg = {"d": 96, "nhead": 4, "layers": 3, "dropout": 0.1, "lr": 3e-4, "wd": 1e-4,
       "batch": 256, "epochs": 120, "patience": 20}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
files = sorted((repo / "data" / "minimum_bias").glob("matched_*.root"))

D = build(files, 3, 100.0, selector=lambda c: select_knn(c, 25))
toks = D["tok_seed"]; y = D["y"]; Et = D["Etrue"]; region = D["region"]; agg = D["agg"]
keep = np.flatnonzero((Et >= 1.0) & (Et <= 100.0))
ktr, kva, kte = (keep[s] for s in split(len(keep)))
in_dim = toks[int(keep[0])].shape[1]

G = np.stack([agg[:, 0], agg[:, 3], np.log(agg[:, 2] + 1.0), agg[:, 1], agg[:, 4]], 1).astype(np.float32)
n_global = G.shape[1]
la, lb = np.polyfit(agg[ktr, 0], y[ktr], 1)
base_all = (la * agg[:, 0] + lb).astype(np.float32)

gb = HistGradientBoostingRegressor(max_iter=300, random_state=0).fit(agg[ktr], y[ktr])
BDT = float(resolution(np.exp(gb.predict(agg[kte])), Et[kte])["sigma_eff"])
ta, tb = np.polyfit(np.log(D["total_energy"][ktr] + EPS), y[ktr], 1)
TE = float(resolution(np.exp(ta * np.log(D["total_energy"][kte] + EPS) + tb), Et[kte])["sigma_eff"])
SUMcal = float(resolution(np.exp(la * agg[kte, 0] + lb), Et[kte])["sigma_eff"])

maxL = max(t.shape[0] for t in toks)
N = len(toks)
Xall = np.zeros((N, maxL, in_dim), np.float32)
Mall = np.zeros((N, maxL), np.bool_)
Wall = np.zeros((N, maxL), np.float32)
for i, t in enumerate(toks):
    L = t.shape[0]; Xall[i, :L] = t; Mall[i, :L] = True
    e = np.expm1(np.clip(t[:, 0], 0, None)); Wall[i, :L] = e / (e.sum() + 1e-9)
cont = np.concatenate([toks[i][:, :7] for i in ktr], 0); tmean = cont.mean(0); tstd = cont.std(0) + EPS
Xall[:, :, :7] = (Xall[:, :, :7] - tmean) / tstd
Xall[~Mall] = 0.0
gmean = G[ktr].mean(0); gstd = G[ktr].std(0) + EPS
Gall = ((G - gmean) / gstd).astype(np.float32)

Xt = torch.from_numpy(Xall).to(DEVICE); Mt = torch.from_numpy(Mall).to(DEVICE)
Wt = torch.from_numpy(Wall).to(DEVICE); Gt = torch.from_numpy(Gall).to(DEVICE)
Bt = torch.from_numpy(base_all).unsqueeze(1).to(DEVICE)
Yt = torch.from_numpy(y.astype(np.float32)).unsqueeze(1).to(DEVICE)
{"BDT": round(BDT, 4), "total_energy": round(TE, 4), "sum_calib": round(SUMcal, 4),
 "n_train": int(len(ktr)), "n_test": int(len(kte)), "clusters": int(N), "device": DEVICE, "maxL": int(maxL)}

{'BDT': 0.1253,
 'total_energy': 0.3583,
 'sum_calib': 0.1837,
 'n_train': 57056,
 'n_test': 12227,
 'clusters': 89797,
 'device': 'cuda',
 'maxL': 25}

## Why adaptive binning 
Plot the true-energy spectrum: it is far from uniform, so fixed-width bins put very few events in the high-energy bins and the per-bin resolution there gets noisy. Quantile bins fix that.

In [2]:
import plotly.graph_objects as go
n_bins = 8
edges = np.quantile(Et[kte], np.linspace(0, 1, n_bins + 1))
fig = go.Figure(go.Histogram(x=Et[kte], nbinsx=60, marker_color="#4c78a8"))
for e in edges:
    fig.add_vline(x=e, line_dash="dot", line_color="crimson", opacity=0.6)
fig.update_layout(template="plotly_white", height=380,
                  title="True photon energy spectrum (test set) with quantile-bin edges (red)",
                  xaxis_title="E_true [GeV]", yaxis_title="clusters")
fig.show()
{"quantile_edges_GeV": [round(float(e), 1) for e in edges]}

{'quantile_edges_GeV': [1.2, 9.1, 14.3, 19.2, 24.6, 32.2, 43.2, 62.6, 100.0]}

In [3]:
def train_eval(make_model, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = make_model().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    def batches(idx, bs, shuffle):
        idx = np.asarray(idx)
        if shuffle:
            idx = rng.permutation(idx)
        for j in range(0, len(idx), bs):
            b = torch.from_numpy(idx[j:j + bs]).to(DEVICE)
            yield Xt[b], Mt[b], Wt[b], Gt[b], Bt[b], Yt[b]

    def vloss():
        model.eval(); s = 0.0; c = 0
        with torch.no_grad():
            for X, m, w, g, base, yb in batches(kva, 512, False):
                s += nn.functional.mse_loss(model(X, m, w, g, base), yb).item(); c += 1
        return s / max(c, 1)

    best = 1e9; bstate = None; wait = 0
    for ep in range(cfg["epochs"]):
        model.train()
        for X, m, w, g, base, yb in batches(ktr, cfg["batch"], True):
            opt.zero_grad()
            nn.functional.mse_loss(model(X, m, w, g, base), yb).backward(); opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4:
            best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break
    model.load_state_dict(bstate); model.eval()
    a, b = np.polyfit(predict_raw(model, kva), y[kva], 1)
    pe = np.exp(a * predict_raw(model, kte) + b)
    return float(resolution(pe, Et[kte])["sigma_eff"]), model, (float(a), float(b))


def predict_raw(model, idx):
    idx = np.asarray(idx); out = []
    with torch.no_grad():
        for j in range(0, len(idx), 512):
            b = torch.from_numpy(idx[j:j + 512]).to(DEVICE)
            out.append(model(Xt[b], Mt[b], Wt[b], Gt[b], Bt[b]).cpu().numpy().ravel())
    return np.concatenate(out)

In [4]:
def encoder():
    layer = nn.TransformerEncoderLayer(cfg["d"], cfg["nhead"], dim_feedforward=4 * cfg["d"],
                                       dropout=cfg["dropout"], batch_first=True)
    return nn.TransformerEncoder(layer, cfg["layers"], enable_nested_tensor=False)

def head(nf):
    return nn.Sequential(nn.Linear(nf, cfg["d"]), nn.ReLU(), nn.Dropout(cfg["dropout"]), nn.Linear(cfg["d"], 1))

class Base(nn.Module):
    def __init__(self, extra=0):
        super().__init__()
        self.embed = nn.Linear(in_dim, cfg["d"]); self.enc = encoder()
        self.norm = nn.LayerNorm(cfg["d"]); self.head = head(cfg["d"] + extra)
    def encode(self, x, m):
        return self.enc(self.embed(x), src_key_padding_mask=~m)

class MeanDirect(Base):
    def __init__(self): super().__init__(extra=n_global)
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m); wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return self.head(torch.cat([p, g], 1))

class MeanResidual(Base):
    def __init__(self): super().__init__(extra=n_global)
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m); wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))

class EfnResidual(Base):
    def __init__(self): super().__init__(extra=n_global)
    def forward(self, x, m, w, g, base):
        h = self.encode(x, m)
        return base + self.head(torch.cat([self.norm((h * w.unsqueeze(-1)).sum(1)), g], 1))

The **only** difference between `MeanDirect` and `MeanResidual` is the `base +` term - identical encoder, pooling, and head. 

In [5]:
VARIANTS = {"MeanDirect (true-E target)": MeanDirect,
            "MeanResidual (bias target)": MeanResidual,
            "EfnResidual (bias target)": EfnResidual}
res13 = {}; models = {}; calib = {}; rows = []
for name, cls in VARIANTS.items():
    vals = []
    for s in range(SEEDS):
        sig, mdl, ab = train_eval(cls, s)
        vals.append(sig)
        if s == 0:
            models[name] = mdl; calib[name] = ab
    mean, std = float(np.mean(vals)), float(np.std(vals))
    res13[name] = vals
    rows.append({"variant": name, "sigma_eff": round(mean, 4), "std": round(std, 4),
                 "BDT": round(BDT, 4), "beats_BDT": mean < BDT})
    print(f"{name}: {mean:.4f} +/- {std:.4f} {'BEATS' if mean < BDT else 'loses'} BDT {BDT:.4f}", flush=True)
summary13 = pd.DataFrame(rows).sort_values("sigma_eff").reset_index(drop=True)
summary13

MeanDirect (true-E target): 0.0714 +/- 0.0020 BEATS BDT 0.1253


MeanResidual (bias target): 0.0723 +/- 0.0001 BEATS BDT 0.1253


EfnResidual (bias target): 0.0645 +/- 0.0024 BEATS BDT 0.1253


,variant,sigma_eff,std,BDT,beats_BDT
0,EfnResidual (bias target),0.0645,0.0024,0.1253,True
1,MeanDirect (true-E target),0.0714,0.0020,0.1253,True
2,MeanResidual (bias target),0.0723,0.0001,0.1253,True


In [6]:
d = summary13
colors = ["#2ca02c" if b else "#8c8c8c" for b in d["beats_BDT"]]
fig = go.Figure(go.Bar(x=d["variant"], y=d["sigma_eff"], error_y=dict(type="data", array=d["std"]),
                       marker_color=colors, text=[f"{v:.4f}" for v in d["sigma_eff"]], textposition="outside"))
fig.add_hline(y=BDT, line_dash="dash", line_color="crimson", annotation_text=f"fair BDT {BDT:.4f}", annotation_position="top left")
fig.add_hline(y=SUMcal, line_dash="dot", line_color="darkorange", annotation_text=f"sum-calib {SUMcal:.4f}", annotation_position="bottom left")
fig.update_layout(template="plotly_white", height=450, title="Min-bias R (overall) resolution: two targets vs baselines",
                  yaxis_title="sigma_eff", xaxis_tickangle=-15)
fig.show()

## Resolution vs energy - adaptive (quantile) bins 
For the best model, the BDT, and the raw calibrated sum, compute `sigma_eff` inside each equal-count energy bin. Adaptive bins keep the high-energy points from going noisy.

In [7]:
def sigma_vs_energy(pred_e, true_e, edges):
    cx, sy, ny = [], [], []
    for i in range(len(edges) - 1):
        hi = edges[i + 1] + (1e-6 if i == len(edges) - 2 else 0.0)
        m = (true_e >= edges[i]) & (true_e < hi)
        if m.sum() >= 20:
            cx.append(float(np.median(true_e[m])))
            sy.append(float(resolution(pred_e[m], true_e[m])["sigma_eff"]))
            ny.append(int(m.sum()))
    return np.array(cx), np.array(sy), ny

best_name = summary13.iloc[0]["variant"]
ma, mb = calib[best_name]
pe_best = np.exp(ma * predict_raw(models[best_name], kte) + mb)
pe_bdt = np.exp(gb.predict(agg[kte]))
pe_sum = np.exp(la * agg[kte, 0] + lb)

curves = {best_name: pe_best, "BDT": pe_bdt, "sum-calib": pe_sum}
cols = {best_name: "#2ca02c", "BDT": "#8c8c8c", "sum-calib": "#d62728"}
fig = go.Figure()
for name, pe in curves.items():
    cx, sy, ny = sigma_vs_energy(pe, Et[kte], edges)
    fig.add_trace(go.Scatter(x=cx, y=sy, mode="lines+markers", name=name, line=dict(color=cols[name])))
fig.update_layout(template="plotly_white", height=430,
                  title="Resolution vs true energy (adaptive quantile bins, ~equal count)",
                  xaxis_title="E_true [GeV] (bin median)", yaxis_title="sigma_eff")
fig.show()

## Read-out
Compare against the clean-signal result in nb12:
- **Target:** if `MeanResidual` (bias) still beats `MeanDirect` (true-E) here, the correction-target choice transfers to the realistic sample. If the gap **widens** under pileup, the bias framing matters more once background is present.
- **Architecture vs BDT:** on clean signal the transformer beat BDT by a thin margin (0.036 vs 0.039). If that margin is **larger** under min-bias, this is the first evidence that attention earns its keep exactly where Felipe expects - when each cell mixes photon + background energy.
- **Adaptive bins** where in energy the learned model pulls ahead of the calibrated sum tells us which regime the correction helps most.

Caveat still open: `total_energy`/sum here includes background energy, so the sum baseline is no longer as strong as on clean signal - part of any transformer gain is just *not* summing in pileup. Next: per-event with/without matching and feature importance under pileup.